# Notebook 02 — Workflows and Activities

This notebook covers Workflow and Activity definitions, the two core building blocks of Temporal applications.

## What You Will Learn
- How to define a Workflow
- How to define an Activity
- How to execute Activities from Workflows
- Workflow determinism rules

## Workflow Definition

A Workflow is a durable function that orchestrates Activities. Rules:
- Must be **deterministic** (no I/O, no randomness, no system time)
- Decorated with `@workflow.defn` and `@workflow.run`
- Entry method must be `async`

In [ ]:
from temporalio import workflow


@workflow.defn
class GreetSomeone:
    """Simple workflow that returns a greeting."""

    @workflow.run
    async def run(self, name: str) -> str:
        return f"Hello, {name}!"


print("Workflow class defined:", GreetSomeone)

## Activity Definition

Activities perform non-deterministic work: API calls, file I/O, database operations.

### Function-Based Activity

In [ ]:
from temporalio import activity


@activity.defn
async def greet(name: str) -> str:
    """Activity that generates a greeting message."""
    return f"Hello, {name}! Welcome to Temporal."


print("Activity defined:", greet)

### Class-Based Activity

Use classes when Activities need dependencies (HTTP sessions, DB connections):

In [ ]:
import urllib.parse
from temporalio import activity


class TranslateActivities:
    """Activities that call a translation microservice."""

    def __init__(self, session):
        self.session = session

    @activity.defn
    async def greet_in_spanish(self, name: str) -> str:
        base = "http://localhost:9999/get-spanish-greeting"
        url = f"{base}?name={urllib.parse.quote(name)}"
        async with self.session.get(url) as response:
            response.raise_for_status()
            return await response.text()

    @activity.defn
    async def farewell_in_spanish(self, name: str) -> str:
        base = "http://localhost:9999/get-spanish-farewell"
        url = f"{base}?name={urllib.parse.quote(name)}"
        async with self.session.get(url) as response:
            response.raise_for_status()
            return await response.text()


print("Class-based Activities defined:", TranslateActivities)

## Executing Activities from Workflows

### Method 1: `workflow.execute_activity()` — for function-based Activities
```python
result = await workflow.execute_activity(
    greet, name, start_to_close_timeout=timedelta(seconds=5)
)
```

### Method 2: `workflow.execute_activity_method()` — for class-based Activities
```python
result = await workflow.execute_activity_method(
    TranslateActivities.greet_in_spanish, name,
    start_to_close_timeout=timedelta(seconds=5)
)
```

In [ ]:
from datetime import timedelta
from temporalio import workflow


# Workflow that orchestrates two activities
@workflow.defn
class GreetAndFarewellWorkflow:
    @workflow.run
    async def run(self, name: str) -> str:
        greeting = await workflow.execute_activity(
            greet,
            name,
            start_to_close_timeout=timedelta(seconds=5),
        )
        return greeting


print("Workflow with Activity execution defined")

## Workflow Sandbox and Imports

Temporal runs Workflows in a sandbox. Import Activity modules using:

```python
with workflow.unsafe.imports_passed_through():
    from activities import TranslateActivities
```

This bypasses the sandbox for the import without compromising determinism.

## Using Dataclasses for Input/Output

Best practice: use dataclasses for structured data.

In [ ]:
from dataclasses import dataclass


@dataclass
class TranslationInput:
    name: str
    language_code: str


@dataclass
class TranslationOutput:
    hello_message: str
    goodbye_message: str


# Usage
input_data = TranslationInput(name="Pierre", language_code="fr")
print(f"Input: {input_data}")

output_data = TranslationOutput(
    hello_message="Bonjour, Pierre",
    goodbye_message="Au revoir, Pierre"
)
print(f"Output: {output_data}")

## Summary

| Concept | Decorator | Purpose |
|---------|-----------|----------|
| Workflow | `@workflow.defn` + `@workflow.run` | Orchestrate tasks deterministically |
| Activity (function) | `@activity.defn` | Perform external work |
| Activity (class) | `@activity.defn` on methods | External work with dependencies |

**Next:** [03_retry_policies.ipynb](03_retry_policies.ipynb)